# **Attacks on tabular explanations: SHAP background, PDP, counterfactual explanations and fairwashing**

Practice for the module [“Attacks on Explanations”](https://ai-interpretability.school/learn/22/0/0), lessons “Manipulating counterfactual explanations and recourse”, “Fairwashing: rationalizing an unfair model” and “Attacks on the SHAP background and on global explanations”. The scaffold model of Slack et al. (2020) has its own notebook, `PRACTICE_fooling_lime_shap`, and is not repeated here.

All eight attacks run on the same data, the open UCI Adult dataset (the 1994 US Census), so that they can be compared with each other:

| section | attack | what the adversary changes | lesson |
|---|---|---|---|
| 1 | Fool SHAP (Laberge et al., 2023) | weights of the background rows SHAP is computed on | attacks on the SHAP background |
| 2 | Fooling PD (Baniecki et al., 2023) | the data the partial dependence curve is computed on | attacks on the SHAP background |
| 3 | Manipulated CFE (Slack et al., 2021), simplified | the model: it has a backdoor for the counterfactual search | manipulating counterfactual explanations |
| 4 | CFD (Pawelczyk et al., 2023) | nothing: the attacker learns training-set membership from a counterfactual explanation | manipulating counterfactual explanations |
| 5 | LaundryML (Aïvodji et al., 2019) | the explanation: a fair-looking surrogate is picked out of many | fairwashing |
| 6 | ε-constrained fairwashing (Aïvodji et al., 2021) | the explanation: a surrogate with a constraint on unfairness | fairwashing |
| 7 | Off-manifold manipulation and tsp (Anders et al., 2020) | the model off the data manifold; the defense is a tangent-space projection | fairwashing |
| 8 | Explanation loss (Dimanov et al., 2020) | the model: fine-tuning with a penalty on the gradient with respect to a feature | fairwashing |

Each section asks three questions: does the attack work on these data, which hyperparameter controls its strength, and where does it break. Exercises without answer keys close the notebook.

The notebook runs on CPU in three to four minutes.

In [ ]:
!pip install shap==0.52.0 dice-ml==0.12 fairlearn==0.14.0 -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # numpy, sklearn and tqdm print harmless warnings on some platforms
import copy, itertools, math, io, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import shap
import dice_ml
from scipy import sparse, stats
from scipy.optimize import linprog
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import partial_dependence
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from fairlearn.reductions import ExponentiatedGradient, DemographicParity

SEED = 0
rng = np.random.default_rng(SEED)   # every numpy random step of the notebook draws from this generator
torch.manual_seed(SEED)
torch.set_num_threads(4)            # the same numbers on a laptop and in Colab

## UCI Adult data

The training part of the Adult dataset from the UCI Machine Learning Repository (Becker and Kohavi, 1996, license CC BY 4.0, DOI [10.24432/C5XW20](https://doi.org/10.24432/C5XW20)), as in `PRACTICE_fooling_lime_shap`: 32561 people from the 1994 US Census. The target is an annual income above 50 thousand dollars, which is the favorable outcome. The features are age, years of education (`education_num`), hours worked per week (`hours_per_week`), marriage (`married`: 1 for the marital status Married-civ-spouse or Married-AF-spouse), sex and race. The sensitive feature is `race_black`, group $s=1$: people recorded in the census with race Black.

Continuous features are standardized, binary features stay zeros and ones. The data are split three ways: training of the black box (60%), the suing group, people who contest decisions and on whom the company builds its explanation (20%), and the auditor's held-out sample (20%).

In [ ]:
URL = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/adult.csv'
df = pd.read_csv(URL)
raw = pd.DataFrame({
    'age': df.age,
    'education_num': df['education-num'],
    'hours_per_week': df['hours-per-week'],
    'married': df['marital-status'].isin(['Married-civ-spouse', 'Married-AF-spouse']).astype(int),
    'sex_male': (df.sex == 'Male').astype(int),
    'race_black': (df.race == 'Black').astype(int),
})
y = (df.income == '>50K').astype(int).values
features = list(raw.columns)
CONT = ['age', 'education_num', 'hours_per_week']
RACE, AGE = features.index('race_black'), features.index('age')
mu, sd = raw[CONT].mean(), raw[CONT].std()
X = raw.astype(float)
X[CONT] = (X[CONT] - mu) / sd
X = X.values
s = X[:, RACE].astype(int)
idx_tr, idx_rest = train_test_split(np.arange(len(X)), test_size=0.4, random_state=SEED, stratify=y)
idx_sg, idx_ho = train_test_split(idx_rest, test_size=0.5, random_state=SEED, stratify=y[idx_rest])
print(len(X), 'rows; training / suing group / held-out:', len(idx_tr), '/', len(idx_sg), '/', len(idx_ho))
print('share of s=1:', round(s.mean(), 3))

The black box is gradient boosting. Its unfairness is measured by the demographic parity gap: the difference between the shares of the “income above 50K” prediction in groups $s=1$ and $s=0$.

In [ ]:
bb = HistGradientBoostingClassifier(max_iter=200, random_state=SEED).fit(X[idx_tr], y[idx_tr])
f_bb = lambda Z: bb.predict_proba(Z)[:, 1]

def dp_gap(pred, group):
    """Demographic parity gap |P(pred=1 | s=1) - P(pred=1 | s=0)| of 0/1 predictions or probabilities."""
    return abs(pred[group == 1].mean() - pred[group == 0].mean())

print('black-box accuracy on the held-out sample:', round((bb.predict(X[idx_ho]) == y[idx_ho]).mean(), 3))
print('demographic parity gap of predictions on the suing group:', round(dp_gap(bb.predict(X[idx_sg]), s[idx_sg]), 3))

## 1. Fool SHAP: background weights the tests do not catch

The audit scenario of the lesson “Attacks on the SHAP background and on global explanations”. The auditor checks demographic parity and asks which share of the gap is due to race. The foreground $\mathcal F$ is drawn from group $s=0$, the background $\mathcal B$ from group $s=1$; then the global Shapley values sum to $\mathbb E[f\mid s=0]-\mathbb E[f\mid s=1]$, and the value of race $\Phi_s$ is its share of the gap.

The global value is a mean over the background: $\Phi_s=\frac1{|\mathcal B|}\sum_{z\in\mathcal B}\Phi_s(z)$, where $\Phi_s(z)$ is the value of race with a single background point $z$, averaged over the foreground. So if the background rows of $D_1$ are drawn with weights $\omega$, then $\Phi_s(\omega)=\sum_j\omega_j\Phi_s(z^{(j)})$ is linear in the weights (Prop. 4.1 of the paper). First we compute the values of all features with a single background point $z$ for every row of $D_1$ exactly, by enumerating all 64 coalitions of the six features, and check the means against `shap.ExactExplainer`.

In [ ]:
D0, D1 = X[s == 0], X[s == 1]
S0 = D0[rng.choice(len(D0), 100, replace=False)]   # unbiased foreground S'_0, 100 rows of group s=0
n_feat = X.shape[1]
coalitions = [S for k in range(n_feat + 1) for S in itertools.combinations(range(n_feat), k)]

def shapley_per_baseline(F, B):
    """Exact interventional Shapley values of every feature with a single baseline z, averaged over the foreground F.
    Returns an array (len(B), n_feat): one row per baseline row z of B."""
    Fx = np.tile(F, (len(B), 1))
    v = {}
    for S in coalitions:                   # v(S): mean prediction with features S from x and the rest from z
        H = np.repeat(B, len(F), axis=0)
        H[:, list(S)] = Fx[:, list(S)]
        v[S] = f_bb(H).reshape(len(B), len(F)).mean(axis=1)
    phi = np.zeros((len(B), n_feat))
    for S in coalitions:
        for i in range(n_feat):
            if i not in S:
                w = math.factorial(len(S)) * math.factorial(n_feat - len(S) - 1) / math.factorial(n_feat)
                phi[:, i] += w * (v[tuple(sorted(S + (i,)))] - v[S])
    return phi

Phi_z = shapley_per_baseline(S0, D1)          # one row per row z of D1
phi_j = Phi_z[:, RACE]                        # the value of race for every background row
B_unbiased = D1[rng.choice(len(D1), 200, replace=False)]
ex = shap.ExactExplainer(f_bb, shap.maskers.Independent(B_unbiased, max_samples=200))
sv_unbiased = ex(S0, silent=True).values
print('global values by shap.ExactExplainer, background of 200 random rows of D1:', np.round(sv_unbiased.mean(0), 4))
print('the same from the Phi_z table:', np.round(shapley_per_baseline(S0, B_unbiased).mean(0), 4))
print('Phi_s over all of D1 (unbiased background):', round(phi_j.mean(), 4))
print('sum of global values and difference of the mean predictions of the groups:', round(Phi_z.mean(0).sum(), 4), round(f_bb(S0).mean() - f_bb(D1).mean(), 4))

Now the weights. The adversary minimizes $\beta\,\Phi_s(\omega)+\lambda\,\mathcal W(\mathcal B,\mathcal B'_\omega)$, where $\beta$ is the sign of $\Phi_s$ on the unbiased background (Algorithm 1 of the paper: the first term is linear, with no absolute value, so at small $\lambda$ the value can cross zero), and $\mathcal W$ is the Wasserstein distance between the original and the reweighted background **in the space of model predictions**: the auditor's tests check exactly the distribution of predictions. For one-dimensional distributions $\mathcal W$ is the area between the distribution functions, $\sum_k|F_\omega(p_k)-F(p_k)|\,(p_{k+1}-p_k)$ over sorted predictions $p_k$, so the whole problem is a linear program. The paper reduces the same problem to a minimum-cost flow; the answer is the same, the solver is different.

The auditor receives the foreground and a background sample of size $M=200$ drawn with weights $\omega$ and checks them as in Algorithm 2 of the paper: a Kolmogorov–Smirnov test compares the predictions on each subset with the predictions on her own random subsample of the same size, and a Wald test compares them with the mean prediction of the whole group. There are four tests, so the p-value threshold is $\alpha/4$ with $\alpha=0.05$ (the Bonferroni correction). As in Algorithm 3, for every $\lambda$ we count the detection rate over 100 background samples; there the attack is accepted if this rate is below the threshold $\tau=10\%$. Our foreground has 100 rows, while in the paper both subsets have size $M$: the table of $\Phi_s(z)$ for all 3124 rows of $D_1$ is computed by enumerating coalitions, and with the smaller foreground it takes half the time.

In [ ]:
pred1, pred0 = f_bb(D1), f_bb(D0)   # the auditor has the predictions on all of D1 and D0
order = np.argsort(pred1)
gaps = np.diff(pred1[order])
N1 = len(D1)
beta = np.sign(phi_j.mean())         # Algorithm 1 of the paper: the sign of Phi_s on the unbiased background

def fool_weights(lam):
    """Algorithm 1 of the paper: weights on the rows of D1 minimizing beta * sum w*phi + lam * W1(predictions of D1, predictions reweighted by w). A sparse LP."""
    # variables: w (N1, sorted by prediction), c (N1-1 cumulative sums of w), t (N1-1, |c_k - k/N1|)
    nv = N1 + 2 * (N1 - 1)
    cost = np.r_[beta * phi_j[order], np.zeros(N1 - 1), lam * gaps]
    rows, cols, vals = [], [], []
    for k in range(N1 - 1):                     # c_k - c_{k-1} - w_k = 0
        rows += [k, k]; cols += [N1 + k, k]; vals += [1.0, -1.0]
        if k > 0:
            rows.append(k); cols.append(N1 + k - 1); vals.append(-1.0)
    A_eq = sparse.vstack([sparse.csr_matrix((vals, (rows, cols)), shape=(N1 - 1, nv)),
                          sparse.csr_matrix((np.ones(N1), (np.zeros(N1), np.arange(N1))), shape=(1, nv))])
    b_eq = np.r_[np.zeros(N1 - 1), 1.0]
    ref = np.arange(1, N1) / N1
    I, Z = sparse.identity(N1 - 1), sparse.csr_matrix((N1 - 1, N1))
    A_ub = sparse.vstack([sparse.hstack([Z, I, -I]), sparse.hstack([Z, -I, -I])])
    b_ub = np.r_[ref, -ref]
    res = linprog(cost, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=(0, None), method='highs')
    w = np.zeros(N1)
    w[order] = res.x[:N1]
    return w / w.sum()

M, ALPHA, REPS = 200, 0.05, 100

def detect(S0_, S1_, r):
    """Algorithm 2 of the paper for both transmitted subsets: KS against the auditor's own uniform subsample
    of the same size and Wald against all predictions of the group, Bonferroni threshold ALPHA/4.
    Returns 1 if fraud is detected and the four p-values."""
    pv = []
    for S_, pred_all in [(S0_, pred0), (S1_, pred1)]:
        p = f_bb(S_)
        own = r.choice(pred_all, len(p), replace=False)
        pv += [stats.ks_2samp(own, p).pvalue,
               2 * stats.norm.sf(abs(p.mean() - pred_all.mean()) / (pred_all.std() / np.sqrt(len(p))))]
    return int(min(pv) < ALPHA / 4), np.round(pv, 3)

def detection_rate(w):
    """Algorithm 3 of the paper: share of REPS background samples of size M, drawn with weights w, that the auditor flags."""
    r = np.random.default_rng(SEED)
    return np.mean([detect(S0, D1[r.choice(N1, M, p=w)], r)[0] for _ in range(REPS)])

print(f'{"lambda":>7}{"Phi_s(w)":>11}{"sum":>9}{"W1":>8}{"weighted rows":>16}{"detection":>14}')
print(f'{"no weights":>7}{phi_j.mean():>11.4f}{Phi_z.mean(0).sum():>9.4f}{0:>8.4f}{N1:>16}{detection_rate(np.full(N1, 1 / N1)):>14.2f}')
weights = {}
for lam in [1, 3, 10, 30]:
    w = fool_weights(lam)
    weights[lam] = w
    W1 = np.abs(np.cumsum(w[order])[:-1] - np.arange(1, N1) / N1) @ gaps
    print(f'{lam:>7}{w @ phi_j:>11.4f}{(w @ Phi_z).sum():>9.4f}{W1:>8.4f}{(w > 1e-9).sum():>16}{detection_rate(w):>14.2f}')


The larger $\lambda$, the closer the distribution of background predictions stays to the original and the less can be hidden. At $\lambda=1$ the value of race crosses zero ($-0.0009$), all the weight sits on 103 rows, and even then the auditor detects the substitution in only 8% of background samples; on the unbiased background false detections occur in 3%. On these data the tests on predictions do not see the substitution at any $\lambda$ in the table: the detection rate is from 2% to 8%, below the threshold $\tau=10\%$. Take $\lambda=10$ and compare the global values of all features on the unbiased background and under the Fool SHAP weights. The sum of global values is the same in every row of the table: it is the difference of the groups' mean predictions, and the distribution of background predictions holds it in place.

In [ ]:
B_fool = D1[np.random.default_rng(SEED).choice(N1, M, p=weights[10])]   # the sample the auditor receives
print('detected (1/0) and p-values KS, Wald for the foreground and for the background:', detect(S0, B_fool, np.random.default_rng(SEED)))
G_unbiased, G_fool = Phi_z.mean(0), weights[10] @ Phi_z      # exact global values: uniform weights and Fool SHAP weights
sv_fool = shap.ExactExplainer(f_bb, shap.maskers.Independent(B_fool, max_samples=M))(S0, silent=True).values
print('Phi_s: unbiased background', round(G_unbiased[RACE], 4), 'Fool SHAP weights', round(G_fool[RACE], 4),
      'the transmitted 200-row sample (shap.ExactExplainer)', round(sv_fool[:, RACE].mean(), 4))
print('reduction of |Phi_s| by the weights:', f'{1 - abs(G_fool[RACE]) / abs(G_unbiased[RACE]):.0%}')
print('global values, unbiased background:', ', '.join(f'{k} {v:.4f}' for k, v in zip(features, G_unbiased)))
print('global values, Fool SHAP weights:', ', '.join(f'{k} {v:.4f}' for k, v in zip(features, G_fool)))

fig, ax = plt.subplots(figsize=(9, 3.6))
pos = np.arange(n_feat)
ax.bar(pos - 0.2, G_unbiased, 0.4, label='unbiased background (all of D1)', color='#8a8983')
ax.bar(pos + 0.2, G_fool, 0.4, label='Fool SHAP weights, lambda=10', color='#eb6834')
ax.axhline(0, color='black', lw=0.6)
ax.set_xticks(pos, features, rotation=20)
ax.set_title('global Shapley values, foreground s=0, background s=1')
ax.set_ylabel('global Shapley value')
ax.legend()
plt.tight_layout()
plt.show()

The sum of the bars is the same; what moved is how it is split between features: the contribution of race fell almost fourfold (from 0.0176 to 0.0048), that of marriage (`married`) by a third, while the contribution of education (`education_num`) grew by half and that of age almost threefold. The auditor sees the parity gap explained by education and age rather than by race. Even without the attack race explains a small part of the gap: 0.0176 out of 0.1162. The attack is powerless if the auditor draws the background from the original data herself; that is the first exercise at the end of the notebook.

## 2. Fooling PD: poisoned data for partial dependence

The partial dependence curve $PD_c(X,z)=\frac1N\sum_i f(X_i^{c=z})$ averages predictions over a sample in which feature $c$ of every row is replaced by $z$. The adversary who supplies the sample changes the other features and touches neither the model nor feature $c$. The model here is a fully connected network, `sklearn.neural_network.MLPClassifier`, the feature is age, and the curve is computed by `sklearn.inspection.partial_dependence`.

Baniecki et al. propose two algorithms: a genetic one (black-box) and a gradient one (white-box). We take the gradient one: the network's weights are copied into `torch`, and the sample itself is optimized with Adam. The goal is to make the centered curve flat, that is, to show the auditor that age does not affect the prediction. A second term with weight $\mu$ penalizes shifts of the features' means and standard deviations: an attempt to hide the poisoning from a simple check of the data.

How this differs from the paper. There the loss of the targeted attack is $\|PD_c(X)-T\|$ with an arbitrary target curve $T$; a flat curve is a special case of it. The term with $\mu$ is not in the paper, it is our addition. In the paper's experiments the discrete features are kept constant (§5), while here the main run changes all features except age; the paper's variant is Exercise 7.

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=500, random_state=SEED).fit(X[idx_tr], y[idx_tr])
print('network accuracy on the held-out sample:', round(mlp.score(X[idx_ho], y[idx_ho]), 3))

def torch_mlp(m):
    """The same network in torch (float64), for the white-box attack."""
    Ws = [torch.tensor(w, dtype=torch.float64) for w in m.coefs_]
    bs = [torch.tensor(b, dtype=torch.float64) for b in m.intercepts_]
    def fwd(Z):
        for W, b in zip(Ws[:-1], bs[:-1]):
            Z = torch.relu(Z @ W + b)
        return torch.sigmoid(Z @ Ws[-1] + bs[-1]).squeeze(-1)
    return fwd

Xa = X[rng.choice(idx_ho, 400, replace=False)]          # the sample handed to the auditor
pd_orig = partial_dependence(mlp, Xa, [AGE], grid_resolution=15)
grid = pd_orig['grid_values'][0]
fwd = torch_mlp(mlp)
print('the torch copy matches sklearn:', np.abs(fwd(torch.tensor(Xa)).numpy() - mlp.predict_proba(Xa)[:, 1]).max() < 1e-10)

lo, hi = torch.tensor(X.min(0)), torch.tensor(X.max(0))
def pd_t(model, D):
    """Partial dependence on age over the grid, differentiable in the data D."""
    H = D.unsqueeze(0).repeat(len(grid), 1, 1)
    H[:, :, AGE] = torch.tensor(grid).unsqueeze(1)
    return model(H).mean(1)

def poison(model, mu, movable, steps=500, lr=0.05):
    """Adam on the sample: flatten the centered PD curve; columns outside `movable` (and age) stay as they are."""
    D0 = torch.tensor(Xa)
    D = D0.clone().requires_grad_(True)
    opt = torch.optim.Adam([D], lr=lr)
    mask = torch.tensor([j in movable for j in range(n_feat)], dtype=torch.float64)
    for _ in range(steps):
        opt.zero_grad()
        cur = pd_t(model, D)
        L = ((cur - cur.mean()) ** 2).mean() + mu * (((D.mean(0) - D0.mean(0)) ** 2).sum() + ((D.std(0) - D0.std(0)) ** 2).sum())
        L.backward()
        D.grad *= mask
        opt.step()
        with torch.no_grad():
            D.copy_(torch.minimum(torch.maximum(D, lo), hi))   # stay inside the range of the data
    return D.detach().numpy()

rng_of = lambda c: c.max() - c.min()
ALL = [j for j in range(n_feat) if j != AGE]
poisoned = {}
print(f'{"mu":>5}{"PD range":>12}{"max KS over features":>26}')
print(f'{"-":>5}{rng_of(pd_orig["average"][0]):>12.3f}{0:>26.3f}')
for mu_ in [0.0, 1.0, 10.0]:
    Xp = poison(fwd, mu_, ALL)
    poisoned[mu_] = Xp
    cur = partial_dependence(mlp, Xp, [AGE], grid_resolution=15)['average'][0]
    ks = max(stats.ks_2samp(Xa[:, j], Xp[:, j]).statistic for j in range(n_feat))
    print(f'{mu_:>5}{rng_of(cur):>12.3f}{ks:>26.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
age_years = grid * sd['age'] + mu['age']
ax.plot(age_years, pd_orig['average'][0], color='#8a8983', lw=2, label='original sample')
for mu_, col in [(0.0, '#eb6834'), (10.0, '#2a78d6')]:
    ax.plot(age_years, partial_dependence(mlp, poisoned[mu_], [AGE], grid_resolution=15)['average'][0], color=col, lw=2, label=f'poisoned sample, mu={mu_:g}')
ax.set_xlabel('age, years')
ax.set_ylabel('partial dependence:\nmean P(income > 50K)')
ax.legend()
plt.tight_layout()
plt.show()

At $\mu=0$ the attack flattens the whole curve: the range shrinks from 0.329 to 0.005, and on the picture age does not affect the prediction. The penalty $\mu$ on means and spreads weakens the attack: the range is 0.131 at $\mu=1$ and 0.161 at $\mu=10$, half the original, and the curve still rises with age. The penalty reduces the distribution shifts only a little: the largest Kolmogorov–Smirnov statistic over the features falls from 0.630 to 0.475 and 0.463. If the auditor has the original data, such poisoning is visible; the paper assumes she does not.

Where the attack breaks entirely is a linear model. The centered partial dependence of the logistic regression logit equals $w_c(z-\bar z)$ and does not depend on the other features, so there is nothing to poison.

In [ ]:
logreg = LogisticRegression(max_iter=1000).fit(X[idx_tr], y[idx_tr])
w_lr, b_lr = torch.tensor(logreg.coef_[0]), torch.tensor(logreg.intercept_[0])
Xp_lr = poison(lambda Z: Z @ w_lr + b_lr, 0.0, ALL)      # the logit of the logistic regression
a = partial_dependence(logreg, Xa, [AGE], grid_resolution=15, response_method='decision_function')['average'][0]
b = partial_dependence(logreg, Xp_lr, [AGE], grid_resolution=15, response_method='decision_function')['average'][0]
print('logistic regression: largest change of the centered curve', f'{np.abs((a - a.mean()) - (b - b.mean())).max():.1e}')

## 3. Manipulated CFE: a backdoor for the counterfactual search

The Wachter method searches for a counterfactual explanation by descent on $G(x,x_{cf})=\lambda\,(f(x_{cf})-1)^2+d(x,x_{cf})$ from a starting point, and the point found is the local minimum nearest to the start. Slack et al. train a model so that the descent from the application $x$ itself ends in a distant minimum, and from a point shifted by a secret $\delta$, in a near one.

This is a **simplified version**: we build the backdoor by hand instead of learning it by implicit differentiation, as the paper does. The mechanism is the same: the result of the search depends on the starting point. The model $m$ coincides with an ordinary network $f$ everywhere except narrow regions of the favorable outcome at half-integer ages for group $s=0$. Age in the data is an integer, so $m=f$ on every application. The secret shift is $\delta$ = half a year of age: from $x+\delta$ the search starts inside such a region. The distance $d$ of Wachter et al. and of the paper is $\ell_1$ weighted by $1/\mathrm{MAD}$; here it is the squared $\ell_2$ distance on the standardized scale. With $\ell_1/\mathrm{MAD}$ and a constant $\lambda=5$ the descent barely leaves the start: Wachter et al. increase $\lambda$ during the search for this, and here the distance does not affect the mechanism of the backdoor.

The favorable outcome is an income above 50K. The search changes years of education and hours worked per week; age, marriage, sex and race are immutable. The cost is the $\ell_1$ distance to the original application on the standardized scale.

In [ ]:
fav = y
MUT = torch.tensor([0., 1., 1., 0., 0., 0.])     # education and hours may move; age, marital status, sex and race are fixed
torch.manual_seed(SEED)
f_net = torch.nn.Sequential(torch.nn.Linear(n_feat, 64), torch.nn.Tanh(), torch.nn.Linear(64, 64), torch.nn.Tanh(), torch.nn.Linear(64, 1))
Xtr_t, fav_t = torch.tensor(X[idx_tr], dtype=torch.float32), torch.tensor(fav[idx_tr], dtype=torch.float32)
opt = torch.optim.Adam(f_net.parameters(), lr=1e-2)
for _ in range(300):
    opt.zero_grad()
    torch.nn.functional.binary_cross_entropy_with_logits(f_net(Xtr_t).squeeze(1), fav_t).backward()
    opt.step()

REGION_H, REGION_W = 12.0, 0.05                  # height (logit) and width (years) of the backdoor regions
def backdoor_region(Z):
    years = Z[:, AGE] * sd['age'] + mu['age']
    frac = years - torch.floor(years) - 0.5        # 0 at half-years, +-0.5 at whole years
    return torch.exp(-frac ** 2 / (2 * REGION_W ** 2)) * (1 - Z[:, RACE])
f_logit = lambda Z: f_net(Z).squeeze(1)
m_logit = lambda Z: f_net(Z).squeeze(1) + REGION_H * backdoor_region(Z)

def wachter(logit, start, steps=200, lr=0.05, lam=5.0):
    """Gradient descent on lam*(sigmoid(f(z))-1)^2 + ||z - start||^2 from `start`, over the mutable features."""
    z = start.clone().detach()
    for _ in range(steps):
        z.requires_grad_(True)
        G = lam * (torch.sigmoid(logit(z)) - 1) ** 2 + ((z - start) ** 2).sum(1)
        z = (z - lr * torch.autograd.grad(G.sum(), z)[0] * MUT).detach()
    return z

Xho_t = torch.tensor(X[idx_ho], dtype=torch.float32)
with torch.no_grad():
    print('largest difference of P(>50K) between m and f on the held-out sample:', (torch.sigmoid(f_logit(Xho_t)) - torch.sigmoid(m_logit(Xho_t))).abs().max().item())
    neg = (f_logit(Xho_t) < 0).numpy()
xs = Xho_t[neg]
prot = (xs[:, RACE] > 0.5).numpy()
delta = torch.zeros(n_feat)
delta[AGE] = 0.5 / sd['age']

def recourse_cost(logit, start):
    cf = wachter(logit, start)
    with torch.no_grad():
        valid = (torch.sigmoid(logit(cf)) > 0.5).numpy()
    return (cf - xs).abs().sum(1).numpy(), valid

noise = torch.randn(xs.shape, generator=torch.Generator().manual_seed(SEED)) * 0.1
print(len(xs), 'applications with an unfavorable prediction; mean recourse cost:')
print(f'{"":<34}{"s=1":>7}{"s=0":>7}{"valid":>9}')
for tag, logit, start in [('f, start at x', f_logit, xs), ('m, start at x (audit)', m_logit, xs),
                          ('m, start at x+delta (owner)', m_logit, xs + delta),
                          ('m, random start x+noise', m_logit, xs + noise),
                          ('m, x+delta+noise', m_logit, xs + delta + noise)]:
    c, ok = recourse_cost(logit, start)
    print(f'{tag:<34}{c[prot].mean():>7.3f}{c[~prot].mean():>7.3f}{ok.mean():>9.2f}')

The audit from real applications sees exactly the same costs for $m$ as for $f$. The model's owner adds $\delta$ and gets recourse almost 17 times cheaper for group $s=0$ (0.037 against 0.618), and only for it. For $f$ the search finds a counterfactual explanation for only 26% of applications: with two mutable features an income above 50K is not reachable for everyone, while from $x+\delta$ the search on $m$ finds one for 91%. A random start of the search, the first defense in §5.3 of the paper, removes the advantage: from $x+\delta+$noise the search almost never lands in a narrow region. In the paper the start noise is $\mathcal N(0,1)$; here its standard deviation is 0.1 on the standardized scale, that is 1.4 years of age, which is enough to leave regions 0.05 years wide. The noise of the immutable features stays in the point found and enters the cost, so the rows with noise are more expensive.

Now an audit with a library. `dice_ml` with the `random` method searches for counterfactual explanations without gradients and finds the same ones for $f$ and $m$: it does not touch the immutable age, and the narrow regions are hidden exactly in age. In the paper the backdoor is learned and has no such simple signature; here a query to the model with a fractional age gives it away, which is the last line of the next cell's output.

In [ ]:
class TorchModel:
    """dice_ml calls predict_proba on raw (unstandardized) rows."""
    def __init__(self, logit):
        self.logit = logit
    def predict_proba(self, D):
        D = pd.DataFrame(D)[features].astype(float)
        D[CONT] = (D[CONT] - mu) / sd
        with torch.no_grad():
            p = torch.sigmoid(self.logit(torch.tensor(D.values, dtype=torch.float32))).numpy()
        return np.c_[1 - p, p]
    def predict(self, D):
        return (self.predict_proba(D)[:, 1] > 0.5).astype(int)

dice_data = dice_ml.Data(dataframe=raw.assign(fav=fav), continuous_features=CONT, outcome_name='fav')
queries = raw.iloc[idx_ho][neg].iloc[:60]

def dice_costs(logit, vary):
    exp = dice_ml.Dice(dice_data, dice_ml.Model(model=TorchModel(logit), backend='sklearn'), method='random')
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):   # dice_ml prints a progress bar
        res = exp.generate_counterfactuals(queries, total_CFs=1, desired_class=1, features_to_vary=vary, random_seed=SEED, verbose=False)
    cost = np.full(len(queries), np.nan)            # nan: dice_ml found no counterfactual for this query
    for i, (q, e) in enumerate(zip(queries.values, res.cf_examples_list)):
        if e.final_cfs_df is not None:
            cf = e.final_cfs_df[features].values.astype(float)[0]
            cost[i] = np.abs((cf - q) / np.r_[sd.values, 1, 1, 1]).sum()
    return cost

VARY = ['education_num', 'hours_per_week']
qprot = queries.race_black.values == 1
for tag, logit in [('f', f_logit), ('m', m_logit)]:
    c = dice_costs(logit, VARY)
    print(f'dice_ml random, {tag}: found {(~np.isnan(c)).sum()} of {len(c)}; mean cost s=1 {np.nanmean(c[qprot]):.3f}, s=0 {np.nanmean(c[~qprot]):.3f}')
with torch.no_grad():
    flips = (m_logit(xs + delta) > 0).numpy()
print('share of s=0 applications where m says income >50K at age +0.5 years:', round(flips[~prot].mean(), 3),
      'for s=1:', round(flips[prot].mean(), 3))

## 4. CFD: a counterfactual explanation reveals training-set membership

Training pushes the decision boundary away from training points, so the nearest counterfactual explanation of a member is on average farther away than that of a point the model has not seen. The CFD attack declares an object a member if this distance exceeds a threshold; the attack's quality is the area under the ROC curve over all thresholds. For a linear model the nearest counterfactual explanation is the projection onto the hyperplane, and the distance to it is $|w^\top x+b|/\|w\|$.

First Adult: a logistic regression is trained on $n$ members, and their distances are compared with those of as many non-members. Then synthetic data with a growing number of features, as in §5.3 of the paper.

In [ ]:
def cfd_auc(Xa_, ya_, n, seed=SEED):
    """ROC AUC of the CFD attack against a logistic regression trained on n members; n non-members are held out."""
    perm = np.random.default_rng(seed).permutation(len(Xa_))
    mem, non = perm[:n], perm[n:2 * n]
    m = LogisticRegression(C=1.0, max_iter=5000).fit(Xa_[mem], ya_[mem])
    dist = np.abs(m.decision_function(Xa_)) / np.linalg.norm(m.coef_)
    return roc_auc_score(np.r_[np.ones(n), np.zeros(n)], np.r_[dist[mem], dist[non]])

SEEDS = range(10)   # one split is noisy: every AUC below is a mean over 10 random member/non-member splits
for n in [50, 200, 1000]:
    print(f'Adult, n={n:>4}: AUC {np.mean([cfd_auc(X, y, n, sd_) for sd_ in SEEDS]):.3f}')

dims = [5, 20, 50, 100, 200, 400]
def synthetic(d, seed=SEED):
    r = np.random.default_rng(seed)
    Xs_ = r.normal(size=(1000, d))
    ys_ = (Xs_ @ r.normal(size=d) / np.sqrt(d) + 0.5 * r.normal(size=1000) > 0).astype(int)
    return Xs_, ys_
auc_d = {n: [np.mean([cfd_auc(*synthetic(d, sd_), n, sd_) for sd_ in SEEDS]) for d in dims] for n in [100, 200]}
for n in auc_d:
    print(f'synthetic, n={n}:', ' '.join(f'd={d}: {a:.3f}' for d, a in zip(dims, auc_d[n])))

fig, ax = plt.subplots(figsize=(5.5, 3.2))
for n, col in [(100, '#eb6834'), (200, '#2a78d6')]:
    ax.plot(dims, auc_d[n], 'o-', color=col, label=f'synthetic, n={n} members')
ax.axhline(0.5, color='#8a8983', ls='--', lw=1, label='random guessing')
ax.set_xscale('log')
ax.set_xlabel('number of features d')
ax.set_ylabel('CFD attack ROC AUC,\nmean over 10 splits')
ax.legend()
plt.tight_layout()
plt.show()

On Adult with six features the attack is at the level of random guessing (AUC from 0.497 to 0.531), as the authors found on Adult. The leakage grows with the number of features once it becomes comparable to the number of training points: the model then fits individual rows, and they push the boundary away.

## 5. LaundryML: enumerating surrogates and picking a convenient one

The company explains the black box by a surrogate that reproduces its predictions on the suing group. LaundryML enumerates many interpretable surrogates and sorts them by the objective $(1-\beta)\,\mathrm{misc}+\beta\cdot\mathrm{unfairness}+\lambda K$: $\mathrm{misc}$ is the share of disagreements with the black box, $\mathrm{unfairness}$ is the surrogate's demographic parity gap, $K$ is its length.

In the paper the surrogates are rule lists from CORELS. Here they are replaced by decision trees of depth up to 4 without the race feature, on every subset of the other features, and for every tree **all** labelings of its leaves are enumerated: the tree splits people into groups, the labeling says which group gets which prediction. Every such model reads like a rule list. Fidelity and unfairness are computed from counts in the leaves, so enumerating hundreds of thousands of models takes seconds.

In [ ]:
Xs_sg, s_sg, b_sg = X[idx_sg], s[idx_sg], bb.predict(X[idx_sg])
Xs_ho, s_ho, b_ho = X[idx_ho], s[idx_ho], bb.predict(X[idx_ho])
nonrace = [j for j in range(n_feat) if j != RACE]

def leaf_counts(tree, cols, Xe, be, se, leaves):
    """Per leaf: black-box positives, negatives and the leaf's contribution to the parity gap."""
    M = tree.apply(Xe[:, cols])[:, None] == leaves[None]
    pos, neg_ = (M & (be[:, None] == 1)).sum(0), (M & (be[:, None] == 0)).sum(0)
    a = (M & (se[:, None] == 1)).sum(0) / (se == 1).sum() - (M & (se[:, None] == 0)).sum(0) / (se == 0).sum()
    return pos, neg_, a, len(be)

labellings = {}
surr = []
for k in range(1, len(nonrace) + 1):
    for cols in itertools.combinations(nonrace, k):
        for depth in [1, 2, 3, 4]:
            t = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=20, random_state=SEED).fit(Xs_sg[:, cols], b_sg)
            leaves = np.unique(t.apply(Xs_sg[:, cols]))
            if len(leaves) not in labellings:
                labellings[len(leaves)] = np.array(list(itertools.product([0, 1], repeat=len(leaves))), dtype=float)
            L = labellings[len(leaves)]
            p, q, a, n_ = leaf_counts(t, cols, Xs_sg, b_sg, s_sg, leaves)
            ph, qh, ah, nh = leaf_counts(t, cols, Xs_ho, b_ho, s_ho, leaves)
            surr.append(dict(cols=cols, depth=depth, K=len(leaves), fid=(L @ p + (1 - L) @ q) / n_, unf=np.abs(L @ a),
                             fid_ho=(L @ ph + (1 - L) @ qh) / nh, unf_ho=np.abs(L @ ah)))
FID, UNF = np.concatenate([r['fid'] for r in surr]), np.concatenate([r['unf'] for r in surr])
FID_HO, UNF_HO = np.concatenate([r['fid_ho'] for r in surr]), np.concatenate([r['unf_ho'] for r in surr])
KK = np.concatenate([np.full(len(r['fid']), r['K']) for r in surr])
print(len(surr), 'trees,', len(FID), 'surrogates')
print('black box: parity gap on the suing group', round(dp_gap(b_sg, s_sg), 3), 'on the held-out sample', round(dp_gap(b_ho, s_ho), 3))

print(f'\n{"beta":>5}{"fidelity":>10}{"unfairness":>12}{"fid. held-out":>15}{"unf. held-out":>16}')
for beta in [0.0, 0.3, 0.4, 0.45, 0.5]:
    i = ((1 - beta) * (1 - FID) + beta * UNF + 0.001 * KK).argmin()
    print(f'{beta:>5}{FID[i]:>10.3f}{UNF[i]:>12.3f}{FID_HO[i]:>15.3f}{UNF_HO[i]:>16.3f}')

The objective with a single weight $\beta$ yields only a few options: up to $\beta=0.3$ the most faithful surrogate wins, from $\beta=0.4$ a surrogate with fidelity 0.916 and unfairness 0.089, and up to $\beta=0.5$ the choice does not change. The intermediate options show up if the question is asked differently: what is the highest fidelity achievable at unfairness at most $u$. That is the fidelity-unfairness front, and the picture shows it next to all the enumerated surrogates.

In [ ]:
def best_under(u):
    ok = UNF <= u
    return np.where(ok)[0][FID[ok].argmax()]
print(f'{"u":>6}{"fidelity":>10}{"unfairness":>12}{"fid. held-out":>15}{"unf. held-out":>16}')
front = []
for u in [0.3, 0.2, 0.15, 0.1, 0.08, 0.05, 0.02]:
    i = best_under(u)
    front.append(i)
    print(f'{u:>6}{FID[i]:>10.3f}{UNF[i]:>12.3f}{FID_HO[i]:>15.3f}{UNF_HO[i]:>16.3f}')

fig, ax = plt.subplots(figsize=(6, 3.6))
sub = np.random.default_rng(SEED).choice(len(FID), 20000, replace=False)
ax.scatter(UNF[sub], FID[sub], s=2, color='#c9c8c3', label='enumerated surrogates (20000 of them)')
ax.plot(UNF[front], FID[front], 'o-', color='#eb6834', label='front on the suing group')
ax.plot(UNF_HO[front], FID_HO[front], 'o--', color='#2a78d6', label='same surrogates, held-out sample')
ax.axvline(dp_gap(b_sg, s_sg), color='black', lw=0.8, ls=':', label='black-box parity gap')
ax.set_xlabel('demographic parity gap of the surrogate')
ax.set_ylabel('fidelity to the black box')
ax.set_ylim(0.6, 1.0)
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

At a parity gap three times smaller than the black box's ($u=0.05$ against 0.141), the surrogate still agrees with it on 86% of the suing group (fidelity 0.861). For $u$ from 0.15 to 0.3 the constraint does not bind: the most faithful surrogate is already less unfair than the black box (0.132). On the held-out sample fidelity is nearly the same, while unfairness is higher: 0.063 against 0.050 at $u=0.05$ and 0.027 against 0.018 at $u=0.02$. The surrogate is fitted to the suing group, and on 6512 people the fit gives itself away only in unfairness. That is the weak point the authors name themselves, and the one the next paper tests.

## 6. ε-constrained fairwashing: it generalizes, it transfers, and a risk measure

Aïvodji et al. (2021) pose a constrained problem: the highest fidelity of the explainer to the black box at unfairness on the suing group at most $\epsilon$. The explainer is a logistic regression, and, as in the paper, the problem is solved by the exponentiated gradient (Agarwal et al., 2018) of the `fairlearn` library with the constraint `DemographicParity(difference_bound=ε)`. In `fairlearn` this constraint is not on the difference of the two groups' prediction rates but on the deviation of each group's rate from the overall rate; with the share of group $s=1$ near 0.1 the gap between the two groups comes out at most about $1.1\,\epsilon$. The result is a randomized classifier, a mixture of several logistic regressions; its labels are drawn with a fixed seed.

Three checks from the paper: does the substitution hold on the held-out sample, does the explainer transfer to other black boxes, and how wide is the range of unfairness among models of nearly equal fidelity (the risk measure via the Rashomon set; here the set is the surrogates enumerated in section 5).

In [ ]:
def eps_explainer(eps):
    """As in the paper: logistic regression fitted to the black-box labels on the suing group by the exponentiated
    gradient of fairlearn (Agarwal et al., 2018) under the demographic parity constraint with bound eps."""
    eg = ExponentiatedGradient(LogisticRegression(max_iter=1000), constraints=DemographicParity(difference_bound=eps))
    eg.fit(Xs_sg, b_sg, sensitive_features=s_sg)
    return lambda Z: eg.predict(Z, random_state=SEED)   # the result is a randomized classifier; a fixed seed draws its labels

def fid_unf(expl, Xe, be, se):
    p = expl(Xe)
    return (p == be).mean(), dp_gap(p, se)

print(f'{"eps":>6}{"fidelity":>10}{"unfairness":>12}{"fid. held-out":>15}{"unf. held-out":>16}')
for eps in [0.3, 0.2, 0.15, 0.1, 0.05]:
    e = eps_explainer(eps)
    print(f'{eps:>6}' + ''.join(f'{v:>{w}.3f}' for v, w in zip(fid_unf(e, Xs_sg, b_sg, s_sg) + fid_unf(e, Xs_ho, b_ho, s_ho), [10, 12, 15, 16])))

In [ ]:
e05 = eps_explainer(0.05)
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEED).fit(X[idx_tr], y[idx_tr])
print('transfer of the explainer (eps=0.05), held-out sample:')
for name, model in [('boosting (fitted to it)', bb), ('random forest', rf), ('MLP network of section 2', mlp)]:
    pb = model.predict(Xs_ho)
    fid, unf = fid_unf(e05, Xs_ho, pb, s_ho)
    print(f'  {name:<32} fidelity {fid:.3f}, explainer unfairness {unf:.3f}, black box {dp_gap(pb, s_ho):.3f}')

print('\nrisk measure: range of unfairness among the section-5 surrogates with fidelity at least the best minus delta')
for dlt in [0.01, 0.03, 0.05, 0.1]:
    R = FID >= FID.max() - dlt
    print(f'  delta={dlt}: {R.sum():>6} models, unfairness {UNF[R].min():.3f}..{UNF[R].max():.3f}')

For $\epsilon$ from 0.15 to 0.3 the constraint does not bind: the logistic regression gives a gap of 0.121 even without it. At $\epsilon=0.05$ the gap on the suing group is 0.058, and fidelity drops only from 0.931 to 0.927.

The substitution generalizes and transfers: the explainer fitted to the boosting model is nearly as faithful to the forest (0.911) and the network (0.917), and its unfairness of 0.075 is almost half that of the three black boxes (0.130–0.153). That is why a check based on a drop of fidelity on a second sample catches only some substitutions.

The risk measure depends on the model class and the tolerance. Among the trees of section 5 within one point of fidelity from the best, the gap is from 0.129 to 0.139, nearly as large as the black box's; at a tolerance of 3 points the range already includes a surrogate with a gap of 0.084, and at 10 points one with 0.043. On these data the measure warns of fairwashing risk already at a small tolerance. The logistic regression with $\epsilon=0.05$ belongs to another class, and the measure over trees does not see it. The measure says how far to distrust an explanation within the chosen class, but does not catch a specific substitution.

## 7. Off-manifold manipulation and tsp explanations

The theorem of Anders et al.: for any model $g$ there is a $\tilde g$ that coincides with $g$ on the data manifold and has arbitrary saliency maps, because derivatives along directions orthogonal to the manifold do not affect predictions on the data.

Tabular data acquire such a manifold by themselves under one-hot encoding: the five race columns sum to 1, and so do the two sex columns and the two marriage columns. The direction $n=(1,\dots,1)$ inside the race block is orthogonal to the manifold. Take
$$\tilde g(x)=g(x)+(n^\top x-1)\,h(x),\qquad h(x)=-\sum_{r}x_r\,\frac{\partial g}{\partial x_r}(x),$$
with the sum over the race columns. On the data $n^\top x=1$, so $\tilde g=g$. The gradient changes: $\nabla\tilde g=\nabla g+h(x)\,n$, and the contribution of race to Gradient×Input, $\sum_r x_r\,\partial_r\tilde g$, becomes zero for every person.

The defense is a tsp explanation (tangent space projection): project the gradient onto the tangent space of the manifold. For one-hot blocks this means subtracting the mean gradient inside every block.

In [ ]:
races = ['White', 'Black', 'Asian-Pac-Islander', 'Amer-Indian-Eskimo', 'Other']
OH = np.c_[X[:, :3], X[:, 3], 1 - X[:, 3], X[:, 4], 1 - X[:, 4], np.stack([df.race.values == r for r in races], 1).astype(float)]
RB = list(range(7, 12))                      # race block
BLOCKS = [[3, 4], [5, 6], RB]                # one-hot blocks: married, sex, race
GROUPS = [[0], [1], [2], [3, 4], [5, 6], RB]   # features as the auditor reads them
GNAMES = CONT + ['married', 'sex', 'race']

def train_net(Xt, yt, seed=SEED, epochs=200, hidden=32):
    torch.manual_seed(seed)
    net = torch.nn.Sequential(torch.nn.Linear(Xt.shape[1], hidden), torch.nn.ReLU(), torch.nn.Linear(hidden, hidden), torch.nn.ReLU(), torch.nn.Linear(hidden, 1))
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    Xt, yt = torch.tensor(Xt, dtype=torch.float32), torch.tensor(yt, dtype=torch.float32)
    for _ in range(epochs):
        opt.zero_grad()
        torch.nn.functional.binary_cross_entropy_with_logits(net(Xt).squeeze(1), yt).backward()
        opt.step()
    return net

g = train_net(OH[idx_tr], y[idx_tr])
g_fn = lambda Z: g(Z).squeeze(1)
n_race = torch.zeros(OH.shape[1])
n_race[RB] = 1.0

def g_tilde(Z):
    """Equals g wherever the race dummies sum to 1; its gradient on the person's own race dummy is zero."""
    out = g_fn(Z)
    grad = torch.autograd.grad(out.sum(), Z, create_graph=True)[0]
    h = -(Z[:, RB] * grad[:, RB]).sum(1)
    return out + (Z @ n_race - 1) * h

def gradient(model, Z):
    Z = torch.tensor(Z, dtype=torch.float32, requires_grad=True)
    return torch.autograd.grad(model(Z).sum(), Z)[0].numpy()

def tsp(gr):
    """Tangent-space projection for one-hot blocks: remove each block's mean gradient."""
    gr = gr.copy()
    for b in BLOCKS:
        gr[:, b] -= gr[:, b].mean(1, keepdims=True)
    return gr

def top1_share(gr):
    """Share of people whose largest |Gradient x Input| (summed within a one-hot block) is race."""
    att = gr * OH
    grp = np.stack([att[:, G].sum(1) for G in GROUPS], 1)
    return (np.abs(grp).argmax(1) == GNAMES.index('race')).mean()

Zall = torch.tensor(OH, dtype=torch.float32, requires_grad=True)
print(f'largest difference of the logits of g and g~ over all {len(OH)} people:', (g_fn(Zall) - g_tilde(Zall)).abs().max().item())
gr_g, gr_t = gradient(g_fn, OH), gradient(g_tilde, OH)
race_gi = lambda gr: np.abs((gr[:, RB] * OH[:, RB]).sum(1)).mean()
print(f'{"":<24}{"race contrib.":>18}{"race ranked 1st":>20}')
for tag, gr in [('g', gr_g), ('g~', gr_t), ('tsp(g)', tsp(gr_g)), ('tsp(g~)', tsp(gr_t))]:
    print(f'{tag:<24}{race_gi(gr):>18.3f}{top1_share(gr):>20.3f}')
print('largest difference of the tsp gradients of g and g~:', f'{np.abs(tsp(gr_g) - tsp(gr_t)).max():.1e}')

The models coincide on every person in the data, and Gradient×Input of $\tilde g$ shows no race for anyone. After the tangent-space projection the explanations of the two models coincide up to rounding error, and race comes back: its contribution is 0.289 against 0.359 for the original gradient. Race rarely ranks first even for the original model (for 1.1% of people), so on Adult the attack erases a feature that is rarely the main one anyway. For one-hot blocks the tangent space is known exactly; for images it has to be estimated, which is the limitation of the defense the lesson “Fairwashing: rationalizing an unfair model” talks about.

## 8. Explanation loss: hiding a feature from several explainers at once

Dimanov et al. fine-tune a model with a penalty on the gradient of the loss with respect to the sensitive feature's column: $L'=L+\frac{\alpha}{n}\|\nabla_{X_{:,j}}L\|_1$. Take a network on the six features, fine-tune it with several values of $\alpha$, and for every model compute on the held-out sample:

- the share of people for whom race is among the top two features by gradient magnitude (Gradients) and by SHAP in the Expected Gradients form (`shap.GradientExplainer`);
- the accuracy and the share of predictions that agree with the model fine-tuned without the penalty;
- the demographic parity gap of the predictions and the share of people whose prediction changes if their race is flipped; these two checks look at predictions, not at the explanation.

In [ ]:
pre = train_net(X[idx_tr], y[idx_tr])
Xtr6, ytr6 = torch.tensor(X[idx_tr], dtype=torch.float32), torch.tensor(y[idx_tr], dtype=torch.float32)
Xho6 = torch.tensor(X[idx_ho], dtype=torch.float32)
Xho6_flip = Xho6.clone()
Xho6_flip[:, RACE] = 1 - Xho6_flip[:, RACE]
bg = Xtr6[np.random.default_rng(SEED).choice(len(idx_tr), 100, replace=False)]

def finetune(alpha, epochs=200):
    net = copy.deepcopy(pre)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for _ in range(epochs):
        opt.zero_grad()
        Xg = Xtr6.clone().requires_grad_(True)
        L = torch.nn.functional.binary_cross_entropy_with_logits(net(Xg).squeeze(1), ytr6)
        gx = torch.autograd.grad(L, Xg, create_graph=True)[0]   # L is a mean, so this is (1/n) grad of the per-row losses
        (L + alpha * gx[:, RACE].abs().sum()).backward()
        opt.step()
    return net

def in_top2(att):
    return (np.argsort(-np.abs(att), 1)[:, :2] == RACE).any(1).mean()

print(f'{"alpha":>6}{"Grad top2":>11}{"EG top2":>9}{"acc.":>8}{"agree":>8}{"DP gap":>9}{"race flip":>13}')
for alpha in [0.0, 0.3, 1.0, 3.0, 10.0]:
    net = finetune(alpha)
    if alpha == 0.0:
        ref = net
    with torch.no_grad():
        p = (net(Xho6).squeeze(1) > 0).numpy()
        p_ref = (ref(Xho6).squeeze(1) > 0).numpy()
        p_flip = (net(Xho6_flip).squeeze(1) > 0).numpy()
    Xg = Xho6.clone().requires_grad_(True)
    grad = torch.autograd.grad(net(Xg).sum(), Xg)[0].numpy()
    torch.manual_seed(SEED)
    eg = np.asarray(shap.GradientExplainer(net, bg).shap_values(Xho6, nsamples=50)).reshape(len(Xho6), -1)
    print(f'{alpha:>6}{in_top2(grad):>11.3f}{in_top2(eg):>9.3f}{(p == y[idx_ho]).mean():>8.3f}{(p == p_ref).mean():>8.3f}'
          f'{dp_gap(p.astype(float), s[idx_ho]):>9.3f}{(p != p_flip).mean():>13.3f}')

Already at $\alpha=0.3$ race leaves the top two features of both the gradient and Expected Gradients, although the penalty was written only on the gradient of the loss. For Expected Gradients race was almost never among the top two even without the penalty (for 0.2% of people), for the gradient it was for 6.3%. Accuracy barely changes, while 1.7–4.4% of predictions disagree with the model without the penalty.

Flipping race now changes the prediction for at most 0.5% of people against 5.5% without the penalty: the model has almost stopped using race directly. The parity gap of predictions, however, shrank only from 0.152 to 0.122–0.126: features correlated with race carry the unfairness, and the explainers do not show it. Hence the defense from the lesson “Fairwashing: rationalizing an unfair model”: check fairness on predictions on the data, not on attributions.

## Comparing the eight attacks

| attack | what the adversary needs | what the auditor sees | what catches it in this notebook |
|---|---|---|---|
| Fool SHAP | the right to choose the background | a smaller contribution of race, the tests on predictions are silent | a background the auditor samples herself (exercise 6) |
| Fooling PD | the right to choose the data for PDP | a flat curve | comparison with the original data; a linear model has no attack |
| Manipulated CFE | the model and knowledge of the search algorithm | equal recourse costs | a random start of the search; a query with a fractional age |
| CFD | an (object, explanation) pair | — | on six features the attack is at the level of random guessing |
| LaundryML | black-box predictions on the suing group | a fair surrogate | higher unfairness on the held-out sample, nearly the same fidelity |
| ε-fairwashing | the same | the same, and it transfers to other models | a fidelity drop catches only some substitutions |
| tsp | the model | Gradient×Input without race | tangent-space projection |
| explanation loss | the model and fine-tuning | race outside the top two features of two explainers | the parity gap of predictions |

Attacks at different moments of intervention are caught by different checks, and no single check covers all eight.

## Exercises

Exercises without answer keys: the answers are in the output of your cells and in your reasoning. They are numbered from 6: tasks 1–5 of the lesson “Attacks on tabular explanations: Practice” are in the lesson itself.

**Exercise 6. The auditor draws the background herself.** Draw 20 background samples of 200 rows from $D_1$ without weights (generator `np.random.default_rng(1)`) and compute $\Phi_s$ for each through the `phi_j` table. What range do the values on unbiased backgrounds span, and does the value for the Fool SHAP background at $\lambda=10$ fall into it?

In [ ]:
# Your code here

**Your answer:**

**Exercise 7. Fooling PD without fractional binary features.** The poisoning of section 2 makes binary features fractional, which is easy to notice. Repeat the attack with $\mu=0$, allowing changes only to `education_num` and `hours_per_week`. How much did the curve's range shrink compared with the attack on all features?

In [ ]:
# Your code here

**Your answer:**

**Exercise 8. DiCE with a mutable age.** Run `dice_costs` for model $m$ with `'age'` added to the mutable features and compare the groups' mean costs with the same run for $f$. If there is no difference, explain why the random search of DiCE does not hit regions 0.05 years wide.

In [ ]:
# Your code here

**Your answer:**

**Exercise 9. The price of hiding race.** From the table of section 8: at which smallest $\alpha$ among those tried does race leave the top two features of both explainers, and what percentage of predictions then disagree with the model without the penalty? Why does the parity gap of predictions barely shrink?

**Your answer:**

## What to take away

- **The data an explanation is computed on are part of the explanation.** Fool SHAP and Fooling PD leave the model alone: choosing the background or the sample is enough. Fool SHAP keeps the distribution of background predictions and therefore passes tests on predictions; the defense is a background the auditor draws from the original data herself.
- **The result of a counterfactual search depends on its start.** A backdoor reached only through the shift $\delta$ is invisible to an audit from real applications; a random start closes it.
- **An explanation can work against the people in the data,** but on Adult with six features membership inference from a counterfactual explanation is at the level of random guessing: the leakage grows with the number of features.
- **Fidelity to the black box does not tell a fair surrogate from an unfair one.** Among surrogates of similar fidelity there are fair-looking ones, and they transfer to other black boxes.
- **The freedom of the gradient off the data manifold** lets one erase a feature from Gradient×Input without changing a single prediction; the tangent-space projection brings it back.
- **Fairness is checked on predictions, not on attributions:** a penalty on the gradient hides race from two explainers, while the parity gap of predictions remains.